In [1]:
!pip -q install faiss-cpu sentence-transformers

In [2]:
from pathlib import Path
from google.colab import drive

# Mount Google Drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:

# 1. INSTALL AND IMPORT LIBRARIES

import json
import numpy as np
import pandas as pd
import faiss
import torch



# Main project folder
PROJECT_DIR = Path("/content/drive/MyDrive/NLP Project")

# Folder containing generated embeddings
EMBED_DIR = PROJECT_DIR / "data/embeddings/bge_small"

# Folder for FAISS indexes
INDEX_DIR = (
    PROJECT_DIR / "vector_database/faiss/bge_small"
)

INDEX_DIR.mkdir(parents=True, exist_ok=True)

# Four chunking strategies
STRATEGIES = [
    "fixed_size",
    "sentence_based",
    "recursive",
    "section_aware"
]

# Embedding model used previously
MODEL_NAME = "BAAI/bge-small-en-v1.5"
EMBEDDING_DIM = 384

print("FAISS version:", faiss.__version__)
print("Embedding folder:", EMBED_DIR)
print("Index folder:", INDEX_DIR)

# Check required files before processing
for strategy in STRATEGIES:

    folder = EMBED_DIR / strategy

    for filename in [
        "embeddings.npy",
        "metadata.jsonl",
        "config.json"
    ]:
        path = folder / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Missing file: {path}"
            )

print("\nAll required embedding files were found.")


FAISS version: 1.16.0
Embedding folder: /content/drive/MyDrive/NLP Project/data/embeddings/bge_small
Index folder: /content/drive/MyDrive/NLP Project/vector_database/faiss/bge_small

All required embedding files were found.


Load embeddings and build all four FAISS indexes

In [4]:

# 2. BUILD FAISS INDEXES FOR ALL FOUR STRATEGIES

index_results = []
reference_articles = None

for strategy in STRATEGIES:

    print("\n" + "=" * 60)
    print("BUILDING INDEX:", strategy.upper())
    print("=" * 60)

    # 1. Load embeddings and metadata

    input_folder = EMBED_DIR / strategy

    embeddings = np.load(
        input_folder / "embeddings.npy"
    )

    metadata = pd.read_json(
        input_folder / "metadata.jsonl",
        lines=True
    )

    with open(
        input_folder / "config.json",
        "r",
        encoding="utf-8"
    ) as file:
        embedding_config = json.load(file)

    # 2. Validate embedding configuration

    assert embedding_config["model_name"] == MODEL_NAME, (
        f"{strategy}: incorrect embedding model"
    )

    assert embedding_config["normalized"] is True, (
        f"{strategy}: embeddings are not normalized"
    )

    if embeddings.ndim != 2:
        raise ValueError(
            f"{strategy}: embeddings must be a 2D array"
        )

    num_vectors, dimension = embeddings.shape

    if num_vectors == 0:
        raise ValueError(f"{strategy}: empty embeddings")

    if num_vectors != len(metadata):
        raise ValueError(
            f"{strategy}: embedding and metadata row "
            "counts do not match"
        )

    if dimension != EMBEDDING_DIM:
        raise ValueError(
            f"{strategy}: expected {EMBEDDING_DIM} dimensions, "
            f"got {dimension}"
        )

    # 3. Validate metadata alignment

    if not metadata["chunk_id"].is_unique:
        raise ValueError(
            f"{strategy}: duplicate chunk IDs"
        )

    expected_ids = np.arange(num_vectors)

    actual_ids = metadata["vector_id"].to_numpy(
        dtype=np.int64
    )

    if not np.array_equal(actual_ids, expected_ids):
        raise ValueError(
            f"{strategy}: vector ID alignment error"
        )

    if not metadata["page_numbers"].map(
        lambda pages: isinstance(pages, list)
        and len(pages) > 0
    ).all():
        raise ValueError(
            f"{strategy}: invalid page references"
        )

    # Ensure all strategies cover the same articles
    current_articles = set(metadata["article_id"])

    if reference_articles is None:
        reference_articles = current_articles

    elif current_articles != reference_articles:
        raise ValueError(
            "Article coverage differs across strategies"
        )

    # 4. Prepare vectors for FAISS

    # FAISS requires float32 vectors
    vectors = np.ascontiguousarray(
        embeddings,
        dtype=np.float32
    )

    if not np.isfinite(vectors).all():
        raise ValueError(
            f"{strategy}: invalid embedding values"
        )

    # Verify normalization for cosine similarity
    norms = np.linalg.norm(vectors, axis=1)

    if not np.allclose(norms, 1.0, atol=1e-3):
        raise ValueError(
            f"{strategy}: vectors are not unit-normalized"
        )

    # 5. Create FAISS index

    # Inner product equals cosine similarity
    # when vectors are normalized.
    index = faiss.IndexFlatIP(dimension)

    # Add embeddings in metadata row order
    index.add(vectors)

    # Check the number of indexed vectors
    assert index.ntotal == num_vectors

    # Verify stored vector alignment
    assert np.allclose(
        index.reconstruct(0),
        vectors[0],
        atol=1e-6
    )

    # 6. Save FAISS index

    output_folder = INDEX_DIR / strategy
    output_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    index_file = output_folder / "index.faiss"

    faiss.write_index(
        index,
        str(index_file)
    )

    # 7. Save index configuration

    index_config = {
        "strategy": strategy,
        "index_type": "IndexFlatIP",
        "similarity": "cosine",
        "embedding_model": MODEL_NAME,
        "embedding_dimension": dimension,
        "normalized": True,
        "total_vectors": num_vectors,
        "total_articles": len(current_articles),
        "metadata_file": str(
            input_folder / "metadata.jsonl"
        ),
        "embedding_file": str(
            input_folder / "embeddings.npy"
        )
    }

    with open(
        output_folder / "index_config.json",
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(index_config, file, indent=4)

    # 8. Store summary

    index_results.append({
        "Strategy": strategy,
        "Total Vectors": index.ntotal,
        "Articles": len(current_articles),
        "Dimensions": dimension,
        "Index Type": "IndexFlatIP",
        "Status": "Saved"
    })

    print("Vectors indexed:", index.ntotal)
    print("Dimensions:", dimension)
    print("Saved:", index_file)

    # Free resources
    del embeddings, metadata, vectors, index


# FINAL INDEXING SUMMARY

index_summary = pd.DataFrame(index_results)

index_summary.to_csv(
    INDEX_DIR / "index_summary.csv",
    index=False
)

print("\nFAISS INDEXING COMPLETED")
display(index_summary)



BUILDING INDEX: FIXED_SIZE
Vectors indexed: 5703
Dimensions: 384
Saved: /content/drive/MyDrive/NLP Project/vector_database/faiss/bge_small/fixed_size/index.faiss

BUILDING INDEX: SENTENCE_BASED
Vectors indexed: 5839
Dimensions: 384
Saved: /content/drive/MyDrive/NLP Project/vector_database/faiss/bge_small/sentence_based/index.faiss

BUILDING INDEX: RECURSIVE
Vectors indexed: 6106
Dimensions: 384
Saved: /content/drive/MyDrive/NLP Project/vector_database/faiss/bge_small/recursive/index.faiss

BUILDING INDEX: SECTION_AWARE
Vectors indexed: 7623
Dimensions: 384
Saved: /content/drive/MyDrive/NLP Project/vector_database/faiss/bge_small/section_aware/index.faiss

FAISS INDEXING COMPLETED


,Strategy,Total Vectors,Articles,Dimensions,Index Type,Status
0,fixed_size,5703,180,384,IndexFlatIP,Saved
1,sentence_based,5839,180,384,IndexFlatIP,Saved
2,recursive,6106,180,384,IndexFlatIP,Saved
3,section_aware,7623,180,384,IndexFlatIP,Saved


Verify the saved FAISS indexes

In [5]:
# 3. VERIFY SAVED FAISS INDEXES

for strategy in STRATEGIES:

    index_file = (
        INDEX_DIR / strategy / "index.faiss"
    )

    metadata_file = (
        EMBED_DIR / strategy / "metadata.jsonl"
    )

    # Reload index from Google Drive
    index = faiss.read_index(
        str(index_file)
    )

    # Reload metadata
    metadata = pd.read_json(
        metadata_file,
        lines=True
    )

    # Validate saved index
    assert index.ntotal == len(metadata)
    assert index.d == EMBEDDING_DIM
    assert index.is_trained

    assert np.array_equal(
        metadata["vector_id"].to_numpy(),
        np.arange(len(metadata))
    )

    print("\nStrategy:", strategy)
    print("Total vectors:", index.ntotal)
    print("Dimensions:", index.d)
    print("Index trained:", index.is_trained)
    print("Validation: PASSED")

print("\nAll four FAISS indexes verified successfully.")



Strategy: fixed_size
Total vectors: 5703
Dimensions: 384
Index trained: True
Validation: PASSED

Strategy: sentence_based
Total vectors: 5839
Dimensions: 384
Index trained: True
Validation: PASSED

Strategy: recursive
Total vectors: 6106
Dimensions: 384
Index trained: True
Validation: PASSED

Strategy: section_aware
Total vectors: 7623
Dimensions: 384
Index trained: True
Validation: PASSED

All four FAISS indexes verified successfully.


Test semantic search using your FAISS indexes

In [6]:
# 4. IMPLEMENT BASIC SEMANTIC RETRIEVAL

from sentence_transformers import SentenceTransformer

# Use GPU if available, otherwise CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load the SAME model used for document embeddings
model = SentenceTransformer(
    MODEL_NAME,
    device=device
)

# Recommended BGE query instruction
QUERY_PREFIX = (
    "Represent this sentence for searching "
    "relevant passages: "
)


def semantic_search(question, strategy="recursive", top_k=5):
    """
    Retrieve top-k passages using a saved FAISS index.

    Returns a DataFrame containing:
    - similarity score
    - chunk ID
    - article ID
    - title
    - page numbers
    - passage text
    """

    if strategy not in STRATEGIES:
        raise ValueError("Unknown chunking strategy")

    if not isinstance(question, str) or not question.strip():
        raise ValueError("Question cannot be empty")

    if top_k < 1:
        raise ValueError("top_k must be positive")

    # Load corresponding FAISS index
    index = faiss.read_index(
        str(INDEX_DIR / strategy / "index.faiss")
    )

    # Load corresponding chunk metadata
    metadata = pd.read_json(
        EMBED_DIR / strategy / "metadata.jsonl",
        lines=True
    )

    # Add BGE instruction to the query only
    query_text = QUERY_PREFIX + question.strip()

    # Convert question into a normalized embedding
    query_vector = model.encode(
        [query_text],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype(np.float32)

    query_vector = np.ascontiguousarray(query_vector)

    # Retrieve top-k nearest passages
    k = min(top_k, index.ntotal)

    scores, indices = index.search(
        query_vector,
        k
    )

    results = []

    for rank, (score, vector_id) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):

        if vector_id < 0:
            continue

        # Use the FAISS result to find original metadata
        chunk = metadata.iloc[int(vector_id)]

        results.append({
            "rank": rank,
            "score": round(float(score), 4),
            "chunk_id": chunk["chunk_id"],
            "article_id": chunk["article_id"],
            "title": chunk["title"],
            "year": chunk.get("year"),
            "page_numbers": chunk["page_numbers"],
            "text": chunk["text"]
        })

    return pd.DataFrame(results)


print("Semantic retrieval function initialized.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Semantic retrieval function initialized.


Test all four strategies with the same question

In [7]:
# 5. TEST ALL FOUR FAISS INDEXES

question = (
    "How are graph neural networks used "
    "for financial fraud detection?"
)

TOP_K = 5

for strategy in STRATEGIES:

    print("\n" + "=" * 80)
    print("STRATEGY:", strategy.upper())
    print("=" * 80)

    results = semantic_search(
        question=question,
        strategy=strategy,
        top_k=TOP_K
    )

    for _, row in results.iterrows():

        print("\nRank:", row["rank"])
        print("Similarity:", row["score"])
        print("Article:", row["article_id"])
        print("Title:", row["title"])
        print("Year:", row["year"])
        print("Pages:", row["page_numbers"])

        print("\nRetrieved passage:")
        print(row["text"][:500])

        print("-" * 80)

print("\nSemantic retrieval testing completed.")



STRATEGY: FIXED_SIZE

Rank: 1
Similarity: 0.9057
Article: FD_076
Title: Dynamic Graph Neural Networks for Multi-Level Financial Fraud Detection: A Temporal-Structural Approach
Year: 2024
Pages: [1, 2, 3]

Retrieved passage:
effectiveness of our multi-level approach
for financial fraud detection in dynamic environments.
Keywords: Graph Neural Networks, Financial Fraud Detection, Temporal-Structural Analysis,
Multi-Level Architecture

Introduction
Research Background and Motivation
Financial fraud has emerged as a critical concern for global financial institutions, resulting in
substantial economic losses estimated at $42 billion annually. Traditional fraud detection systems
rely on rule-based approaches and conventi
--------------------------------------------------------------------------------

Rank: 2
Similarity: 0.9016
Article: FD_083
Title: Ensemble of Graph Neural Networks for Enhanced Financial Fraud Detection
Year: 2024
Pages: [1]

Retrieved passage:
. INTRODUCTION
As financial